# Clase 13 — Del notebook a un proyecto ejecutable y responsable
**Taller de Ciencia de Datos (EIN146A25) · 2026-2 · Jueves 5 de noviembre**

Dos preguntas guían la clase: **¿lo puede ejecutar otra persona?** y **¿debería existir con estos datos?** Hay un solo mini-caso: **solicitudes de microcrédito** de una institución ficticia, y el modelo estima **"¿esta persona incumplirá el pago en 12 meses?"**. En los Ejercicios 1 y 2 el análisis pasa de un notebook a un proyecto con scripts, pruebas y un comando único. En los Ejercicios 3 y 4 se audita: riesgo de reidentificación y sesgo por subgrupo. Al final, la prueba del clon vacío sobre su repositorio.

**Datos sintéticos generados para el lab: todas las personas, nombres y correos son inventados; ninguna cifra describe a una institución ni a una persona real.**

| # | Ejercicio | Qué se practica | Min |
|---|---|---|---|
| 1 | De notebook a script que se ejecuta | `src/`, `argparse`, `pathlib`, `logging`, validar y fallar temprano, guardar modelo y métricas | 25 |
| 2 | Pruebas, un comando y requisitos | tres pruebas (`pytest` o `assert`), `run_all.py`, `requirements.txt` con versiones, copia limpia | 20 |
| 3 | Clasificar columnas y medir el riesgo de reidentificación | identificadores, cuasi-identificadores, tamaño de grupo, generalizar y suprimir | 25 |
| 4 | Auditar el sesgo por subgrupo y completar la ficha | falsos positivos y negativos por subgrupo con intervalo, proxies, ficha del modelo | 30 |

Reglas del notebook:
- Los datos **se generan solos** en `data_practica/` y el proyecto de ejemplo en `proyecto_ejemplo/`: no hay que descargar nada y el lab no usa internet.
- Las celdas marcadas **✏️ TU TURNO** son las que deben completar. En los Ejercicios 1 y 2 son celdas `%%writefile`: completan el esqueleto y **ejecutan la celda**, que escribe el archivo dentro de `proyecto_ejemplo/`.
- Cada ejercicio trae su **verificación automática** (debe imprimir todos los ✓). Las soluciones se revisan en clase; la referencia técnica de cada paso está en la guía.
- El Ejercicio 2 usa el `src/` del Ejercicio 1. Los Ejercicios 3 y 4 solo usan lo del Setup (`df`, `pipeline_ref` y las funciones del caso).
- **Si vuelven a ejecutar el Setup, `proyecto_ejemplo/` se borra y se crea de nuevo**: después vuelvan a ejecutar sus celdas `%%writefile`.
- `pytest` es opcional: si no está instalado, el Ejercicio 2 usa la versión con `assert`. Ver `lab13-guia-proyecto-ejecutable-y-responsable.md` para la referencia técnica de cada sección (§N).

In [ ]:
# Setup — ejecutar primero
import ast
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.dpi"] = 110

try:
    import sklearn
except ImportError:
    print("scikit-learn no está instalado: ejecuten  pip install scikit-learn  y reinicien el kernel")
    raise ImportError("Este lab necesita scikit-learn: pip install scikit-learn") from None

import joblib
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    import pytest  # noqa: F401
    HAY_PYTEST = True
except ImportError:
    HAY_PYTEST = False
    print("pytest no está instalado: el Ejercicio 2 usa la versión con assert")

SEED = 2026
CARPETA = Path("data_practica")
CARPETA.mkdir(exist_ok=True)
FIGS = Path("figs")


def guardar_figura(fig, nombre):
    """Guarda una figura en figs/ (la crea si no existe) con la resolución del informe."""
    FIGS.mkdir(exist_ok=True)
    ruta = FIGS / nombre
    fig.savefig(ruta, dpi=150, bbox_inches="tight")
    print("Guardado:", ruta)


def ejecutar_checks(checks):
    """checks: lista de (nombre, funcion_booleana_sin_argumentos).
    Atrapa los errores de lo que todavía no existe para que "Restart & Run All" con TU TURNO vacíos no se caiga."""
    faltan = 0
    for nombre, chk in checks:
        try:
            ok = bool(chk())
        except NameError as e:
            var = str(e).split("'")[1] if "'" in str(e) else str(e)
            print(f"✗ {nombre} (todavía no definen `{var}`)")
        except FileNotFoundError as e:
            print(f"✗ {nombre} (todavía no existe {Path(e.filename).name if e.filename else e})")
        except subprocess.TimeoutExpired:
            print(f"✗ {nombre} (todavía no termina: pasó el límite de 120 s)")
        except Exception as e:
            print(f"✗ {nombre} (error al evaluar: {type(e).__name__}: {e})")
        else:
            print(("✓" if ok else "✗"), nombre)
            faltan += not ok
            continue
        faltan += 1
    print("Todo OK" if faltan == 0 else f"Faltan {faltan}")
    return faltan == 0


def sin_tildes(texto):
    """Minúsculas y sin tildes, para comparar textos con tolerancia."""
    return unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode().lower()


def importados(ruta):
    """Módulos de primer nivel que importa un archivo .py (por ejemplo, 'sklearn' de 'from sklearn.x import y')."""
    arbol_ast = ast.parse(Path(ruta).read_text(encoding="utf-8"))
    mods = set()
    for n in ast.walk(arbol_ast):
        if isinstance(n, ast.Import):
            mods |= {a.name.split(".")[0] for a in n.names}
        elif isinstance(n, ast.ImportFrom) and n.module and n.level == 0:
            mods.add(n.module.split(".")[0])
    return mods


print("Python:", sys.version.split()[0], "· pandas:", pd.__version__, "· scikit-learn:", sklearn.__version__,
      "· pytest:", "sí" if HAY_PYTEST else "no")
print("Carpeta de trabajo:", Path.cwd())

In [ ]:
# Genera el archivo de práctica (no modificar esta celda)
COMUNAS = {  # comuna -> provincia, con peso relativo
    "Concepción": ("Concepción", 18), "Talcahuano": ("Concepción", 12), "Hualpén": ("Concepción", 7),
    "San Pedro de la Paz": ("Concepción", 9), "Chiguayante": ("Concepción", 6), "Coronel": ("Concepción", 7),
    "Lota": ("Concepción", 4), "Penco": ("Concepción", 4), "Tomé": ("Concepción", 4),
    "Los Ángeles": ("Biobío", 13), "Cabrero": ("Biobío", 3), "Mulchén": ("Biobío", 3),
    "Lebu": ("Arauco", 3), "Cañete": ("Arauco", 3),
}
PROVINCIA = {c: v[0] for c, v in COMUNAS.items()}


def generar_solicitudes(n=6000, seed=SEED):
    rng = np.random.default_rng(seed)
    comunas = list(COMUNAS)
    w = np.array([COMUNAS[c][1] for c in comunas], float); w /= w.sum()
    comuna = rng.choice(comunas, n, p=w)
    sexo = rng.choice(["F", "M"], n, p=[0.52, 0.48])
    edad = np.clip(rng.normal(41, 12, n).round(), 18, 74).astype(int)
    # ocupación: servicios_domesticos concentrada en mujeres (proxy de sexo)
    ocup = np.empty(n, dtype=object)
    for i in range(n):
        if sexo[i] == "F":
            p = [0.34, 0.14, 0.17, 0.27, 0.08]
        else:
            p = [0.46, 0.22, 0.24, 0.02, 0.06]
        ocup[i] = rng.choice(["dependiente", "independiente", "comercio_informal", "servicios_domesticos", "jubilado"], p=p)
    base = {"dependiente": 780_000, "independiente": 900_000, "comercio_informal": 520_000,
            "servicios_domesticos": 430_000, "jubilado": 410_000}
    ingreso = np.array([base[o] for o in ocup]) * np.exp(rng.normal(0, 0.28, n))
    ingreso = np.where(sexo == "F", ingreso * 0.93, ingreso)
    ingreso = (np.clip(ingreso, 250_000, 2_500_000) / 1000).round() * 1000
    monto = np.clip(rng.lognormal(np.log(900_000), 0.5, n), 200_000, 3_000_000)
    monto = (monto / 10_000).round() * 10_000
    plazo = rng.choice([6, 12, 18, 24, 36], n, p=[0.15, 0.35, 0.2, 0.2, 0.1])
    dep = np.clip(rng.poisson(1.3, n), 0, 5)
    antig = np.clip((rng.gamma(2.0, 30, n)).round(), 0, 240).astype(int)
    puntaje = np.clip(rng.normal(62, 16, n) + 0.04 * (ingreso / 10_000 - 55), 5, 100).round().astype(int)
    deuda = (rng.random(n) < (0.22 + 0.12 * (ocup == "comercio_informal"))).astype(int)
    dias = rng.integers(0, 365, n)
    fecha = pd.to_datetime("2025-01-01") + pd.to_timedelta(dias, unit="D")

    carga = monto / (ingreso * np.maximum(plazo, 1) / 12)  # cuota-ish relativa al ingreso
    z = lambda x: (x - x.mean()) / x.std()
    logit = (-2.35 + 0.80 * z(carga) - 0.85 * z(puntaje) - 0.45 * z(np.log(ingreso))
             + 0.55 * deuda + 0.25 * (ocup == "comercio_informal") + 0.28 * (sexo == "F")
             - 0.20 * z(antig) + 0.10 * dep)
    p = 1 / (1 + np.exp(-logit))
    incumple = (rng.random(n) < p).astype(int)

    df = pd.DataFrame({
        "id_solicitud": [f"S{i:04d}" for i in range(1, n + 1)],
        "nombre": [f"Solicitante {i:04d}" for i in range(1, n + 1)],
        "correo": [f"s{i:04d}@ejemplo.invalid" for i in range(1, n + 1)],
        "comuna": comuna, "edad": edad, "sexo": sexo,
        "fecha_solicitud": fecha.strftime("%Y-%m-%d"),
        "ocupacion": ocup, "ingreso_mensual": ingreso.astype(int),
        "monto_solicitado": monto.astype(int), "plazo_meses": plazo,
        "n_dependientes": dep, "antiguedad_laboral_meses": antig,
        "puntaje_historial": puntaje, "deuda_previa": deuda,
        "incumple": incumple,
    })
    return df


solicitudes = generar_solicitudes()
solicitudes.to_csv(CARPETA / "solicitudes_microcredito.csv", index=False, encoding="utf-8")
print("solicitudes_microcredito.csv:", solicitudes.shape)
print(f"tasa de incumplimiento: {solicitudes.incumple.mean():.1%}".replace(".", ","))
print("Escrito en", (CARPETA / "solicitudes_microcredito.csv").resolve())

### Setup del caso: modelo, partición y decisión

Esta celda la usan todos los ejercicios.

- **Modelo:** `Pipeline` con `ColumnTransformer` (`StandardScaler` sobre las 8 numéricas y `OneHotEncoder` sobre `ocupacion` y `comuna`) y `LogisticRegression`. **No usa** `sexo`, `id_solicitud`, `nombre`, `correo` ni `fecha_solicitud`.
- **Partición:** 70 % entrenamiento y 30 % test, estratificada, con `random_state=SEED` (4.200 y 1.800 filas).
- **Decisión:** se **niega el crédito** si la probabilidad estimada de incumplimiento es **≥ 0,25** (`UMBRAL`).

> *Un falso positivo es negar el crédito a quien sí habría pagado: lo sufre la persona. Un falso negativo es aprobar a quien incumple: lo asume la institución.*

| Nombre | Qué hace |
|---|---|
| `df` | Las 6.000 solicitudes, leídas desde `data_practica/` |
| `pipeline_generico(num, cat)` | Devuelve el pipeline del caso, sin ajustar, para las columnas dadas |
| `partir(df)` | Devuelve `X_tr, X_te, y_tr, y_te, sexo_te` con la partición del caso (`X` trae todas las columnas menos el objetivo; el pipeline usa solo las suyas) |
| `auc_modelo(df, num, cat)` | Ajusta el pipeline con esas columnas y devuelve el ROC-AUC en test |
| `intervalo_wilson(k, n)` | Intervalo de confianza del 95 % para una tasa `k / n` (guía §11) |
| `pipeline_ref` | El pipeline de referencia **ya ajustado** con `NUMERICAS` y `CATEGORICAS` |

Ver `lab13-guia-proyecto-ejecutable-y-responsable.md`, §11.

In [ ]:
df = pd.read_csv(CARPETA / "solicitudes_microcredito.csv")

NUMERICAS = ["edad", "ingreso_mensual", "monto_solicitado", "plazo_meses",
             "n_dependientes", "antiguedad_laboral_meses", "puntaje_historial", "deuda_previa"]
CATEGORICAS = ["ocupacion", "comuna"]
OBJETIVO = "incumple"
UMBRAL = 0.25


def pipeline_generico(num, cat):
    """Pipeline del caso (escalado + one-hot + regresión logística), sin ajustar."""
    pre = ColumnTransformer([("num", StandardScaler(), num),
                             ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
    return Pipeline([("pre", pre), ("modelo", LogisticRegression(max_iter=1000, random_state=SEED))])


def partir(df):
    """Partición del caso: 70/30 estratificada con SEED. Devuelve X_tr, X_te, y_tr, y_te, sexo_te."""
    X, y = df.drop(columns=[OBJETIVO]), df[OBJETIVO]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=SEED, stratify=y)
    return X_tr, X_te, y_tr, y_te, df.loc[X_te.index, "sexo"]


def auc_modelo(df, num, cat):
    """Ajusta el pipeline con las columnas dadas (partición del caso) y devuelve el ROC-AUC en test."""
    X, y = df[num + cat], df[OBJETIVO]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=SEED, stratify=y)
    return roc_auc_score(y_te, pipeline_generico(num, cat).fit(X_tr, y_tr).predict_proba(X_te)[:, 1])


def intervalo_wilson(k, n, z=1.96):
    """Intervalo de Wilson (95 % con z=1,96) para la tasa k/n. Devuelve (inferior, superior)."""
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h


X_tr, X_te, y_tr, y_te, sexo_te = partir(df)
pipeline_ref = pipeline_generico(NUMERICAS, CATEGORICAS).fit(X_tr, y_tr)
print(f"df: {df.shape[0]} filas × {df.shape[1]} columnas · entrenamiento: {len(X_tr)} · test: {len(X_te)}")

### Setup del proyecto de ejemplo

`crear_proyecto_ejemplo()` arma (o rearma, borrando antes) la carpeta `proyecto_ejemplo/` con la estructura de un repositorio: los datos crudos en `data/raw/`, un paquete `src/` vacío, `tests/` y `models/` vacías y un `requirements.txt` de partida. `ejecutar(...)` corre un comando **desde la raíz del proyecto**, como lo haría otra persona en la terminal.

Ver `lab13-guia-proyecto-ejecutable-y-responsable.md`, §1 y §2.

In [ ]:
RAIZ_PROYECTO = Path("proyecto_ejemplo").resolve()
CMD_EJ1 = [sys.executable, "-m", "src.entrenar", "--datos", "data/raw/solicitudes_microcredito.csv",
           "--salida", "models/", "--semilla", "2026"]


def crear_proyecto_ejemplo():
    """Crea (o recrea) proyecto_ejemplo/ con data/raw, src/, tests/, models/ y un requirements.txt de partida."""
    if RAIZ_PROYECTO.exists():
        shutil.rmtree(RAIZ_PROYECTO)
    for sub in ["data/raw", "src", "tests", "models"]:
        (RAIZ_PROYECTO / sub).mkdir(parents=True)
    shutil.copy(CARPETA / "solicitudes_microcredito.csv", RAIZ_PROYECTO / "data" / "raw")
    (RAIZ_PROYECTO / "src" / "__init__.py").write_text("", encoding="utf-8")
    (RAIZ_PROYECTO / "requirements.txt").write_text("pandas\nscikit-learn\n", encoding="utf-8")


def ejecutar(args, cwd=None, timeout=120):
    """Corre un comando (lista) desde cwd (por defecto, la raíz del proyecto) y devuelve el resultado."""
    env = {**os.environ, "PYTHONIOENCODING": "utf-8", "PYTHONUTF8": "1"}
    return subprocess.run(args, cwd=cwd or RAIZ_PROYECTO, capture_output=True, text=True,
                          encoding="utf-8", errors="replace", timeout=timeout, env=env)


def arbol(raiz, nivel=0):
    """Imprime el árbol de carpetas (sin __pycache__)."""
    raiz = Path(raiz)
    if nivel == 0:
        print(raiz.name + "/")
    for p in sorted(raiz.iterdir()):
        if p.name in {"__pycache__", ".pytest_cache"}:
            continue
        print("    " * (nivel + 1) + p.name + ("/" if p.is_dir() else ""))
        if p.is_dir():
            arbol(p, nivel + 1)


def redirigir(ruta):
    """Solo para la celda 1.1: esa ruta es de otro computador; aquí se lee la copia de data_practica/."""
    return ruta if Path(ruta).exists() else CARPETA / Path(ruta).name


crear_proyecto_ejemplo()
arbol(RAIZ_PROYECTO)

---
## Ejercicio 1 — De notebook a script que se ejecuta (≈25 min)

Un resultado que solo existe en un notebook depende del orden en que se ejecutaron las celdas, del computador y de quién las ejecutó. Un **script** se ejecuta desde la terminal con un comando, recibe sus parámetros desde afuera, revisa sus entradas antes de usarlas y deja sus resultados en archivos.

Ver `lab13-guia-proyecto-ejecutable-y-responsable.md`, §1, §2 y §5.

### 1.1 Primero, el error
El análisis tal como quedó en un notebook, ejecutado **tres veces seguidas** sobre los mismos datos.

In [ ]:
def celda_del_notebook():
    import pandas as pd
    from sklearn.compose import ColumnTransformer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler

    ruta = "C:/Users/alguien/Desktop/solicitudes_microcredito.csv"
    datos = pd.read_csv(redirigir(ruta))   # (redirigir es del Setup: en este computador esa ruta no existe)
    datos = datos.drop(columns=["id_solicitud", "nombre", "correo", "sexo", "fecha_solicitud"])
    X = datos.drop(columns="incumple")
    y = datos["incumple"]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3)
    num = [c for c in X.columns if c not in ("ocupacion", "comuna")]
    pre = ColumnTransformer([("num", StandardScaler(), num),
                             ("cat", OneHotEncoder(handle_unknown="ignore"), ["ocupacion", "comuna"])])
    modelo = Pipeline([("pre", pre), ("lr", LogisticRegression(max_iter=1000))])
    modelo.fit(X_tr, y_tr)
    auc = roc_auc_score(y_te, modelo.predict_proba(X_te)[:, 1])
    return auc


for i in range(1, 4):
    print(f"Ejecución {i}: ROC-AUC = {celda_del_notebook():.4f}".replace(".", ","))

*¿Cuál es el número oficial? ¿Qué pasa si otra persona lo ejecuta en su computador?*

### 1.2 ✏️ TU TURNO
Con la estructura que ya creó `crear_proyecto_ejemplo()`, completen las tres celdas `%%writefile` de abajo (cada una escribe un archivo en `proyecto_ejemplo/src/`) y **ejecútenlas**:

1. **`src/datos.py`**: las constantes `NUMERICAS`, `CATEGORICAS` y `OBJETIVO` (las mismas del Setup); `cargar_datos(ruta)`, que use `pathlib`, falle con `FileNotFoundError` y un mensaje claro si el archivo no existe, y registre el *shape* con `log.info`; y `validar_columnas(df)`, que lance `ValueError("Falta la columna 'x'")` por cada columna esperada que no esté, y que revise que el objetivo sea 0/1 y que las edades sean ≥ 18.
2. **`src/modelo.py`**: `construir_pipeline(semilla)` con `ColumnTransformer` + `LogisticRegression(max_iter=1000)`.
3. **`src/entrenar.py`**: `main()` con `argparse` (`--datos`, `--salida`, `--semilla` con valor por defecto 2026), `logging.basicConfig`, la partición del caso con la semilla, el ajuste, `roc_auc_score` en test, `joblib.dump` del pipeline en `<salida>/pipeline.joblib` y `<salida>/metricas.json` con `auc_test` (redondeado a 3 decimales), `umbral`, `n_train`, `n_test` y `semilla`. Al final, el bloque `if __name__ == "__main__":`.
4. Un **docstring** en cada función: qué hace, qué recibe y cuándo falla.
5. Ejecuten la celda `ejecutar_ej1()` (ya dada): corre `python -m src.entrenar ...` desde la raíz del proyecto y muestra lo que imprime.

In [ ]:
%%writefile proyecto_ejemplo/src/datos.py
# ✏️ TU TURNO — completar y ejecutar esta celda (escribe el archivo)
import logging
from pathlib import Path

import pandas as pd

log = logging.getLogger(__name__)

NUMERICAS = ...    # TODO: las 8 columnas numéricas
CATEGORICAS = ...  # TODO
OBJETIVO = ...     # TODO


def cargar_datos(ruta):
    # TODO: Path(ruta); FileNotFoundError con mensaje claro si no existe; read_csv; log.info con el shape
    raise NotImplementedError


def validar_columnas(df):
    # TODO: ValueError("Falta la columna 'x'") por cada columna esperada; objetivo 0/1; edades >= 18
    raise NotImplementedError

In [ ]:
%%writefile proyecto_ejemplo/src/modelo.py
# ✏️ TU TURNO — completar y ejecutar esta celda (escribe el archivo)
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.datos import CATEGORICAS, NUMERICAS


def construir_pipeline(semilla):
    # TODO: ColumnTransformer (StandardScaler en NUMERICAS, OneHotEncoder en CATEGORICAS) + LogisticRegression
    raise NotImplementedError

In [ ]:
%%writefile proyecto_ejemplo/src/entrenar.py
# ✏️ TU TURNO — completar y ejecutar esta celda (escribe el archivo)
import argparse
import json
import logging
from pathlib import Path

import joblib
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from src.datos import CATEGORICAS, NUMERICAS, OBJETIVO, cargar_datos, validar_columnas
from src.modelo import construir_pipeline

UMBRAL = 0.25


def main():
    # TODO 1: argparse con --datos, --salida y --semilla (por defecto 2026)
    # TODO 2: logging.basicConfig
    # TODO 3: cargar y validar; X, y; partición con la semilla (test_size=0.3, stratify=y)
    # TODO 4: ajustar construir_pipeline(semilla) y calcular roc_auc_score en test
    # TODO 5: guardar pipeline.joblib y metricas.json (auc_test, umbral, n_train, n_test, semilla)
    raise NotImplementedError


if __name__ == "__main__":
    main()

In [ ]:
# (ya dada) Corre el entrenamiento como lo haría otra persona en la terminal, desde la raíz del proyecto
def ejecutar_ej1():
    """Ejecuta src.entrenar con subprocess y muestra la salida y el código de término."""
    print(">> python -m src.entrenar --datos data/raw/solicitudes_microcredito.csv --salida models/ --semilla 2026")
    p = ejecutar(CMD_EJ1)
    if p.stdout.strip():
        print(p.stdout.strip())
    if p.stderr.strip():
        print(p.stderr.strip()[-2000:])
    print("Código de salida:", p.returncode, "(0 = terminó bien)")


ejecutar_ej1()

### ✅ Verificación automática
Ejecuta el script varias veces (≈ 10 s). Debe imprimir 10 ✓.

In [ ]:
def verificar_ej1():
    src = RAIZ_PROYECTO / "src"
    archivos = [src / n for n in ("datos.py", "modelo.py", "entrenar.py")]
    r = {}

    def correr():
        if "p" not in r:
            shutil.rmtree(RAIZ_PROYECTO / "models", ignore_errors=True)
            t = time.time()
            r["p"] = ejecutar(CMD_EJ1)
            r["t"] = time.time() - t
        return r["p"]

    def metricas():
        return json.loads((RAIZ_PROYECTO / "models" / "metricas.json").read_text(encoding="utf-8"))

    def dos_veces():
        m1 = metricas()["auc_test"]
        return ejecutar(CMD_EJ1).returncode == 0 and metricas()["auc_test"] == m1

    def sin_ocupacion():
        with tempfile.TemporaryDirectory(ignore_cleanup_errors=True) as tmp:
            ruta = Path(tmp) / "sin_ocupacion.csv"
            df.drop(columns="ocupacion").to_csv(ruta, index=False, encoding="utf-8")
            p = ejecutar([sys.executable, "-m", "src.entrenar", "--datos", str(ruta), "--salida", str(Path(tmp) / "m")])
        return p.returncode != 0 and "Falta la columna 'ocupacion'" in p.stderr

    def textos():
        return [a.read_text(encoding="utf-8") for a in src.glob("*.py")]

    def funciones():
        return [n for a in src.glob("*.py") for n in ast.walk(ast.parse(a.read_text(encoding="utf-8")))
                if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))]

    checks = [
        ("src/datos.py, src/modelo.py y src/entrenar.py existen y están completos (sin NotImplementedError)",
         lambda: all("NotImplementedError" not in a.read_text(encoding="utf-8") for a in archivos)),
        ("python -m src.entrenar ... termina con código 0 en menos de 120 s",
         lambda: correr().returncode == 0 and r["t"] < 120),
        ("existen models/pipeline.joblib y models/metricas.json",
         lambda: (RAIZ_PROYECTO / "models" / "pipeline.joblib").exists()
                 and (RAIZ_PROYECTO / "models" / "metricas.json").exists()),
        ("metricas.json: auc_test ≈ 0,802 (±0,02), n_train 4200, n_test 1800, semilla 2026",
         lambda: abs(metricas()["auc_test"] - 0.802) <= 0.02 and metricas()["n_train"] == 4200
                 and metricas()["n_test"] == 1800 and metricas()["semilla"] == 2026),
        ("el pipeline guardado carga y predict_proba sobre 5 filas devuelve 5 × 2",
         lambda: joblib.load(RAIZ_PROYECTO / "models" / "pipeline.joblib").predict_proba(df.head(5)).shape == (5, 2)),
        ("ejecutar dos veces da el mismo auc_test",
         dos_veces),
        ("con un CSV sin `ocupacion`, el script falla y avisa \"Falta la columna 'ocupacion'\"",
         sin_ocupacion),
        ("ningún .py de src/ tiene rutas fijas (C:/, C:\\, /Users/, /home/)",
         lambda: not any(s in t for t in textos() for s in ("C:/", "C:\\", "/Users/", "/home/"))),
        ("entrenar.py importa argparse y logging, y src/ usa pathlib",
         lambda: {"argparse", "logging"} <= importados(src / "entrenar.py")
                 and any("pathlib" in importados(a) for a in src.glob("*.py"))),
        ("cada función de src/ tiene docstring",
         lambda: len(funciones()) > 0 and all(ast.get_docstring(f) for f in funciones())),
    ]
    ok = ejecutar_checks(checks)
    if "p" in r and r["p"].returncode != 0:
        print("\nÚltimas líneas del error de src.entrenar:\n" + "\n".join(r["p"].stderr.strip().splitlines()[-4:]))
    return ok

verificar_ej1()

---
## Ejercicio 2 — Pruebas, un comando y requisitos (≈20 min)

"En mi computador funciona" no es una prueba. Para que otra persona confíe en el resultado, el proyecto necesita **pruebas** que fallen si algo se rompe, **un solo comando** que ejecute todo en orden y una lista de **requisitos con versiones**. Este ejercicio usa el `src/` del Ejercicio 1.

Ver `lab13-guia-proyecto-ejecutable-y-responsable.md`, §3, §4 y §7.

### 2.1 Primero, el error
Una copia del proyecto como la recibiría otra persona que lo clona: solo `src/` y `requirements.txt`, sin ejecutar nada antes.

In [ ]:
with tempfile.TemporaryDirectory(ignore_cleanup_errors=True) as tmp:
    clon = Path(tmp) / "clon"
    shutil.copytree(RAIZ_PROYECTO / "src", clon / "src", ignore=shutil.ignore_patterns("__pycache__"))
    shutil.copy(RAIZ_PROYECTO / "requirements.txt", clon)
    arbol(clon)
    p = ejecutar(CMD_EJ1, cwd=clon)

print("\n>> python -m src.entrenar --datos data/raw/solicitudes_microcredito.csv --salida models/ --semilla 2026")
print("Código de salida:", p.returncode)
print("\n".join(p.stderr.strip().splitlines()[-2:]))

*¿Qué le falta al repositorio para que otra persona lo ejecute?*

### 2.2 ✏️ TU TURNO

1. **`tests/test_pipeline.py`** con **tres pruebas** (funciones `test_…`): (a) la salida del preprocesamiento no tiene nulos; (b) el pipeline devuelve una predicción por fila; (c) el ROC-AUC del pipeline supera en más de 0,10 al de un `DummyClassifier(strategy="prior")`. Deben correr de dos formas: con `pytest` (`python -m pytest -q tests`) y sin `pytest`, con un bloque `if __name__ == "__main__":` que llame a las tres (`python -m tests.test_pipeline`). La primera celda de abajo (ya dada) crea `tests/__init__.py`.
2. **`run_all.py`** en la raíz: ejecuta, en orden, el entrenamiento y las pruebas con `subprocess` y `sys.executable`, y termina con código distinto de 0 si algún paso falla.
3. **`requirements.txt`** completo: **con versión** en cada línea (`>=` o `==`) y con **todo** lo que importan `src/` y `tests/`.

In [ ]:
# (ya dada) tests/ como paquete, para poder ejecutar  python -m tests.test_pipeline
_ = (RAIZ_PROYECTO / "tests" / "__init__.py").write_text("", encoding="utf-8")

In [ ]:
%%writefile proyecto_ejemplo/tests/test_pipeline.py
# ✏️ TU TURNO — completar y ejecutar esta celda (escribe el archivo)
# from src.datos import ...
# from src.modelo import construir_pipeline
#
# DATOS = "data/raw/solicitudes_microcredito.csv"
#
# def test_preprocesamiento_sin_nulos(): ...
# def test_una_prediccion_por_fila(): ...
# def test_supera_al_baseline(): ...
#
# if __name__ == "__main__":   # versión sin pytest: python -m tests.test_pipeline
#     ...

In [ ]:
%%writefile proyecto_ejemplo/run_all.py
# ✏️ TU TURNO — completar y ejecutar esta celda (escribe el archivo)
# import subprocess
# import sys
#
# PASOS = [
#     [sys.executable, "-m", "src.entrenar", ...],
#     [sys.executable, "-m", ...],          # las pruebas
# ]
#
# if __name__ == "__main__":
#     ...                                  # ejecutar cada paso; si uno falla, terminar con código distinto de 0

In [ ]:
%%writefile proyecto_ejemplo/requirements.txt
# ✏️ TU TURNO — una línea por paquete, con versión (>= o ==)
pandas
scikit-learn

### ✅ Verificación automática
Ejecuta las pruebas y `run_all.py` dos veces, una en una copia limpia (≈ 15 s). Debe imprimir 5 ✓.

In [ ]:
MAPA_PAQUETES = {"sklearn": "scikit-learn", "joblib": "scikit-learn"}   # joblib viene con scikit-learn


def _requisitos():
    lineas = [l.split("#")[0].strip() for l in
              (RAIZ_PROYECTO / "requirements.txt").read_text(encoding="utf-8").splitlines()]
    return [l for l in lineas if l]


def _faltan_en_requisitos():
    nombres = {re.split(r"[<>=!~\[ ;]", l)[0].lower().replace("_", "-") for l in _requisitos()}
    usados = set()
    for d in ("src", "tests"):
        for f in (RAIZ_PROYECTO / d).glob("*.py"):
            usados |= importados(f)
    terceros = {m for m in usados if m not in sys.stdlib_module_names and m not in {"src", "tests"}}
    return sorted({MAPA_PAQUETES.get(m, m).lower() for m in terceros} - nombres)


def verificar_ej2():
    test = RAIZ_PROYECTO / "tests" / "test_pipeline.py"
    cmd_pruebas = ([sys.executable, "-m", "pytest", "-q", "tests"] if HAY_PYTEST
                   else [sys.executable, "-m", "tests.test_pipeline"])
    fallas = []

    def n_tests():
        return sum(isinstance(n, ast.FunctionDef) and n.name.startswith("test_")
                   for n in ast.parse(test.read_text(encoding="utf-8")).body)

    def correr(args, cwd=None):
        p = ejecutar(args, cwd=cwd)
        if p.returncode != 0:
            fallas.append((" ".join(Path(a).name if i == 0 else a for i, a in enumerate(args)),
                           (p.stdout + p.stderr).strip()))
        return p

    def run_all_ok():
        shutil.rmtree(RAIZ_PROYECTO / "models", ignore_errors=True)
        return (RAIZ_PROYECTO / "run_all.py").exists() and correr([sys.executable, "run_all.py"]).returncode == 0 \
            and (RAIZ_PROYECTO / "models" / "metricas.json").exists()

    def cubre_imports():
        faltan = _faltan_en_requisitos()
        if faltan:
            fallas.append(("requirements.txt", "falta: " + ", ".join(faltan)))
        return not faltan

    def copia_limpia():
        with tempfile.TemporaryDirectory(ignore_cleanup_errors=True) as tmp:
            dst = Path(tmp) / "clon"
            ign = shutil.ignore_patterns("__pycache__", "*.pyc", ".pytest_cache")
            for d in ("src", "tests", "data/raw"):
                shutil.copytree(RAIZ_PROYECTO / d, dst / d, ignore=ign)
            for f in ("run_all.py", "requirements.txt"):
                shutil.copy(RAIZ_PROYECTO / f, dst / f)
            return correr([sys.executable, "run_all.py"], cwd=dst).returncode == 0 \
                and (dst / "models" / "metricas.json").exists()

    checks = [
        ("tests/test_pipeline.py define al menos 3 funciones test_…",
         lambda: n_tests() >= 3),
        ("las pruebas pasan (" + ("python -m pytest -q tests" if HAY_PYTEST else "python -m tests.test_pipeline") + ")",
         lambda: n_tests() >= 3 and correr(cmd_pruebas).returncode == 0),
        ("python run_all.py (sin models/) termina con código 0 y recrea models/metricas.json",
         run_all_ok),
        ("requirements.txt: cada línea con versión (== o >=) y todo lo que importan src/ y tests/",
         lambda: len(_requisitos()) > 0 and all(("==" in l or ">=" in l) for l in _requisitos()) and cubre_imports()),
        ("copia limpia (src, tests, run_all.py, requirements.txt y data/raw): python run_all.py termina con código 0 y crea models/metricas.json",
         copia_limpia),
    ]
    ok = ejecutar_checks(checks)
    for paso, salida in fallas[:2]:
        print(f"\n[{paso}] últimas líneas:\n" + "\n".join(salida.splitlines()[-4:]))
    return ok

verificar_ej2()

### 2.3 Extra (opcional, para la casa) · ✏️ README
Escriban `proyecto_ejemplo/README.md` con la plantilla de la S7 (nueve encabezados): **qué hace** · **requisitos** · **instalación** · **cómo ejecutar** (comandos exactos, en orden) · **estructura** · **datos** · **resultados principales** (cada cifra con su archivo de origen) · **limitaciones** · **uso de IA**. Revisen también que cada función pública tenga docstring. Ver guía §6.

In [ ]:
# ✏️ TU TURNO (opcional): quiten los # y completen
# readme = """# Riesgo de incumplimiento en microcréditos (datos sintéticos)
#
# ## Qué hace
# ...
# ## Requisitos
# ...
# (siete encabezados más)
# """
# (RAIZ_PROYECTO / "README.md").write_text(readme, encoding="utf-8")

### ✅ Verificación automática (extra)
Debe imprimir 3 ✓, o `⊘` si no hicieron el extra (no cuenta como falla).

In [ ]:
ENCABEZADOS_README = ["que hace", "requisitos", "instalacion", "como ejecutar", "estructura", "datos",
                      "resultados", "limitaciones", "uso de ia"]


def verificar_ej2_extra():
    ruta = RAIZ_PROYECTO / "README.md"
    if not ruta.exists():
        print("⊘ extra no realizado (no cuenta como falla)")
        return True
    texto = ruta.read_text(encoding="utf-8")
    titulos = [sin_tildes(l.lstrip("#").strip()) for l in texto.splitlines() if l.startswith("#")]

    def seccion_limitaciones():
        lineas, dentro = [], False
        for l in texto.splitlines():
            if l.startswith("#"):
                dentro = "limitacion" in sin_tildes(l)
            elif dentro and l.strip():
                lineas.append(l.strip())
        return " ".join(lineas)

    checks = [
        ("el README tiene al menos 8 de los 9 encabezados de la plantilla",
         lambda: sum(any(e in t for t in titulos) for e in ENCABEZADOS_README) >= 8),
        ("contiene el comando python run_all.py",
         lambda: "python run_all.py" in texto),
        ("declara al menos una limitación (texto bajo el encabezado de limitaciones)",
         lambda: len(seccion_limitaciones()) >= 20),
    ]
    return ejecutar_checks(checks)

verificar_ej2_extra()

---
## Ejercicio 3 — Clasificar columnas y medir el riesgo de reidentificación (≈25 min)

Antes de modelar con datos de personas hay que saber qué columna identifica a alguien por sí sola, cuál lo identifica combinada con otras, y qué se puede hacer para protegerlas sin perder lo que el análisis necesita.

Ver `lab13-guia-proyecto-ejecutable-y-responsable.md`, §8, §9 y §10.

### 3.1 Primero, el error
Quitamos el nombre. ¿Ya es anónimo?

In [ ]:
anonimo = df.drop(columns=["nombre", "correo", "id_solicitud"])
tam = anonimo.groupby(["comuna", "edad", "sexo", "fecha_solicitud"]).size()
unicas = int((tam == 1).sum())
print("Columnas que quedan:", ", ".join(anonimo.columns))
print(f"Filas cuya combinación comuna + edad + sexo + fecha_solicitud no la comparte nadie más: "
      f"{unicas} de {len(anonimo)} ({100 * unicas / len(anonimo):.1f} %)".replace(".", ","))

*¿Cuántas personas comparten esa combinación?*

### 3.2 ✏️ TU TURNO

1. `clasificacion`: diccionario columna → una de `"identificador_directo"`, `"cuasi_identificador"`, `"financiera"`, `"neutra"`, `"objetivo"` (las 16 columnas); `decision`: columna → una de `"eliminar"`, `"seudonimizar"`, `"generalizar"`, `"conservar"`, `"conservar_solo_auditoria"`; y `justificacion`: columna → una frase.
2. `tam_antes`: `Series` con el tamaño de cada grupo de los cuatro cuasi-identificadores crudos (`comuna`, `edad`, `sexo`, `fecha_solicitud`) con `.size()`; `k_min_antes` (el grupo más pequeño) y `filas_unicas_antes` (filas en grupos de tamaño 1).
3. `df_gen`: copia de `df` con `tramo_edad` (`pd.cut` en los 5 tramos `18-29`, `30-39`, `40-49`, `50-59`, `60-74`), `provincia` (`comuna.map(PROVINCIA)`) y `trimestre` (`pd.PeriodIndex(..., freq="Q").astype(str)`), **sin** `nombre`, `correo`, `id_solicitud`, `comuna`, `edad` ni `fecha_solicitud`. Con los cuasi-identificadores generalizados (`tramo_edad`, `provincia`, `sexo`, `trimestre`): `n_grupos_gen`, `k_min_gen` y `filas_bajo_k` (filas en grupos con menos de `K = 5`).
4. `df_k`: `df_gen` sin las filas de los grupos con menos de 5 (`groupby(...).transform("size")`), y `k_min_final`.
5. `auc_original = auc_modelo(df, NUMERICAS, ["ocupacion", "comuna"])` y `auc_generalizado = auc_modelo(df_gen, <las numéricas sin edad>, ["ocupacion", "provincia", "tramo_edad"])`.
6. En la celda de texto de abajo: una frase sobre cuánto detalle se perdió y qué se ganó.
7. `conclusion_ej3` = `"a"`, `"b"` o `"c"`:
   - a) Quitar nombre y correo basta para anonimizar.
   - b) Combinando comuna, edad, sexo y fecha casi todas las filas son únicas; generalizar y suprimir celdas pequeñas protege, y en este caso casi no cuesta utilidad.
   - c) Generalizar siempre destruye el poder predictivo.

In [ ]:
# ✏️ TU TURNO
# clasificacion = {...}        # las 16 columnas
# decision = {...}
# justificacion = {...}
# tam_antes = ...
# k_min_antes = ...
# filas_unicas_antes = ...
K = 5
# df_gen = ...
# n_grupos_gen = ...
# k_min_gen = ...
# filas_bajo_k = ...
# df_k = ...
# k_min_final = ...
# auc_original = ...
# auc_generalizado = ...
# conclusion_ej3 = ...

✏️ **Respuesta (6):** ¿cuánto detalle se perdió y qué se ganó? *(escriban aquí, con números)*

### ✅ Verificación automática
Debe imprimir 11 ✓.

In [ ]:
CATEGORIAS_COL = {"identificador_directo", "cuasi_identificador", "financiera", "neutra", "objetivo"}
DECISIONES_COL = {"eliminar", "seudonimizar", "generalizar", "conservar", "conservar_solo_auditoria"}


def verificar_ej3():
    cols = set(df.columns)
    checks = [
        ("clasificacion y decision cubren las 16 columnas con valores válidos",
         lambda: set(clasificacion) == cols and set(decision) == cols
                 and set(clasificacion.values()) <= CATEGORIAS_COL and set(decision.values()) <= DECISIONES_COL),
        ("justificacion tiene una frase para cada columna",
         lambda: set(justificacion) == cols and all(len(str(v).strip()) >= 10 for v in justificacion.values())),
        ("nombre, correo e id_solicitud: identificador_directo, y se eliminan o seudonimizan",
         lambda: all(clasificacion[c] == "identificador_directo" and decision[c] in {"eliminar", "seudonimizar"}
                     for c in ["nombre", "correo", "id_solicitud"])),
        ("comuna, edad, sexo y fecha_solicitud: cuasi_identificador",
         lambda: all(clasificacion[c] == "cuasi_identificador" for c in ["comuna", "edad", "sexo", "fecha_solicitud"])),
        ('sexo no se conserva a secas: decision["sexo"] == "conservar_solo_auditoria"',
         lambda: decision["sexo"] == "conservar_solo_auditoria"),
        ("tam_antes suma 6.000; k_min_antes == 1 y filas_unicas_antes ≈ 5.876 (±60)",
         lambda: int(tam_antes.sum()) == len(df) and k_min_antes == 1 and abs(filas_unicas_antes - 5876) <= 60),
        ("generalizado: n_grupos_gen ≈ 120 (±3), k_min_gen ≈ 2 (±1), filas_bajo_k ≈ 20 (±10)",
         lambda: abs(n_grupos_gen - 120) <= 3 and abs(k_min_gen - 2) <= 1 and abs(filas_bajo_k - 20) <= 10),
        ("df_k tiene ≈ 5.980 filas (±10) y k_min_final ≥ 5",
         lambda: abs(len(df_k) - 5980) <= 10 and k_min_final >= 5),
        ("df_k no tiene identificadores ni comuna, edad o fecha_solicitud",
         lambda: not ({"nombre", "correo", "id_solicitud", "comuna", "edad", "fecha_solicitud"} & set(df_k.columns))),
        ("auc_original ≈ 0,802 y auc_generalizado ≈ 0,801 (±0,02)",
         lambda: abs(auc_original - 0.802) <= 0.02 and abs(auc_generalizado - 0.801) <= 0.02),
        ("conclusion_ej3 es la correcta",
         lambda: conclusion_ej3 == "b"),
    ]
    return ejecutar_checks(checks)

verificar_ej3()

---
## Ejercicio 4 — Auditar el sesgo por subgrupo y completar la ficha (≈30 min)

Un modelo puede tener buen desempeño global y equivocarse más con un grupo que con otro. Con la decisión del caso (negar si la probabilidad es ≥ 0,25), un **falso positivo** niega el crédito a quien sí habría pagado y un **falso negativo** aprueba a quien incumple. El modelo `pipeline_ref` **no usa** `sexo`.

Ver `lab13-guia-proyecto-ejecutable-y-responsable.md`, §11.

### 4.1 Primero, el error
La evaluación global del modelo en el test, tal como suele reportarse.

In [ ]:
p_ref = pipeline_ref.predict_proba(X_te)[:, 1]
print(f"ROC-AUC en test: {roc_auc_score(y_te, p_ref):.3f}".replace(".", ","))
print(f"Exactitud con umbral 0,25: {accuracy_score(y_te, (p_ref >= UMBRAL).astype(int)):.3f}".replace(".", ","))
print("¿AUC 0,80? Pregunta: ¿le sirve a todas las personas por igual?")

### 4.2 ✏️ TU TURNO

1. `prediccion_brecha`: **antes de calcular nada**, `"cierra"` o `"no_cierra"`: ¿la diferencia entre sexos desaparece porque el modelo no usa `sexo`? (No se verifica el valor, solo que esté definido.)
2. `tabla_sub`: DataFrame indexado por `sexo` (`F`, `M`) con las columnas `n`, `n_neg`, `n_pos`, `fpr`, `fpr_inf`, `fpr_sup`, `fnr`, `fnr_inf`, `fnr_sup`, calculadas sobre el **test** con `pipeline_ref`, `UMBRAL` e `intervalo_wilson`. `fpr` = negados entre quienes sí pagarían (`n_neg` = quienes sí pagan); `fnr` = aprobados entre quienes incumplen (`n_pos` = quienes incumplen).
3. `brecha_fpr` = `fpr` de F − `fpr` de M, y `se_solapan` (bool): ¿se solapan los intervalos de F y de M para `fpr`?
4. `tabla_sub_con_sexo`: lo mismo con un modelo que **sí** usa `sexo` (`pipeline_generico(NUMERICAS, ["ocupacion", "comuna", "sexo"])` ajustado sobre `X_tr, y_tr`), y `brecha_fpr_con_sexo`.
5. `evidencia_proxy`: DataFrame por `ocupacion` con `n`, `pct_F` (proporción de mujeres, entre 0 y 1), `ingreso_mediano` y `tasa_incumple`, calculado sobre `df`; y `proxy`: una frase (≥ 40 caracteres) sobre qué variable o variables reproducen la brecha.
6. `ficha`: diccionario con las seis claves `uso_previsto_y_no_recomendados`, `datos`, `desempeno`, `limitaciones`, `consideraciones_eticas` y `version_y_fecha`, cada una con un texto, siguiendo la plantilla de la S14 y la guía §11. El desempeño lleva cifras (global y por subgrupo) y la celda de donde salen.
7. `conclusion_ej4` = `"a"`, `"b"` o `"c"`:
   - a) Quitar `sexo` cierra la brecha porque el modelo ya no la ve.
   - b) La brecha persiste porque otras variables (ocupación, ingreso) la reproducen: hay que medir por subgrupo, no solo borrar la columna.
   - c) La brecha no importa porque el AUC global es alto.

In [ ]:
# ✏️ TU TURNO
# prediccion_brecha = ...      # "cierra" o "no_cierra": ANTES de calcular nada
# tabla_sub = ...              # índice sexo; n, n_neg, n_pos, fpr, fpr_inf, fpr_sup, fnr, fnr_inf, fnr_sup
# brecha_fpr = ...
# se_solapan = ...
# tabla_sub_con_sexo = ...
# brecha_fpr_con_sexo = ...
# evidencia_proxy = ...        # por ocupacion: n, pct_F, ingreso_mediano, tasa_incumple
# proxy = ...
# ficha = {
#     "uso_previsto_y_no_recomendados": ...,
#     "datos": ...,
#     "desempeno": ...,
#     "limitaciones": ...,
#     "consideraciones_eticas": ...,
#     "version_y_fecha": ...,
# }
# conclusion_ej4 = ...

### ✅ Verificación automática
Debe imprimir 13 ✓.

In [ ]:
CLAVES_FICHA = ["uso_previsto_y_no_recomendados", "datos", "desempeno", "limitaciones",
                "consideraciones_eticas", "version_y_fecha"]


def _por(t, col):
    return t.set_index(col) if col in t.columns else t


def verificar_ej4():
    def t():
        return _por(tabla_sub, "sexo")

    def ev():
        e = _por(evidencia_proxy, "ocupacion")
        pf = e.loc["servicios_domesticos", "pct_F"]
        return e, (pf / 100 if pf > 1 else pf)

    checks = [
        ('prediccion_brecha es "cierra" o "no_cierra"',
         lambda: prediccion_brecha in {"cierra", "no_cierra"}),
        ("tabla_sub: n = 962 (F) y 838 (M)",
         lambda: int(t().loc["F", "n"]) == 962 and int(t().loc["M", "n"]) == 838),
        ("fpr: F ≈ 0,248 y M ≈ 0,168 (±0,02)",
         lambda: abs(t().loc["F", "fpr"] - 0.248) <= 0.02 and abs(t().loc["M", "fpr"] - 0.168) <= 0.02),
        ("fnr: F ≈ 0,294 y M ≈ 0,412 (±0,03)",
         lambda: abs(t().loc["F", "fnr"] - 0.294) <= 0.03 and abs(t().loc["M", "fnr"] - 0.412) <= 0.03),
        ("cada intervalo contiene a su tasa",
         lambda: all(t().loc[s, f"{m}_inf"] <= t().loc[s, m] <= t().loc[s, f"{m}_sup"]
                     for s in ["F", "M"] for m in ["fpr", "fnr"])),
        ("brecha_fpr ≈ 0,080 (±0,02) y los intervalos no se solapan",
         lambda: abs(brecha_fpr - 0.080) <= 0.02 and not bool(se_solapan)),
        ("con sexo en el modelo la brecha casi no cambia (|diferencia| < 0,02)",
         lambda: abs(brecha_fpr_con_sexo - brecha_fpr) < 0.02),
        ("evidencia_proxy: servicios_domesticos con pct_F ≥ 0,90 e ingreso mediano menor que dependiente",
         lambda: ev()[1] >= 0.90 and ev()[0].loc["servicios_domesticos", "ingreso_mediano"]
                 < ev()[0].loc["dependiente", "ingreso_mediano"]),
        ("proxy: una frase de al menos 40 caracteres que menciona la ocupación o el ingreso",
         lambda: len(proxy.strip()) >= 40 and ("ocupacion" in sin_tildes(proxy) or "ingreso" in sin_tildes(proxy))),
        ("ficha: las seis claves, cada una con al menos 15 caracteres",
         lambda: set(CLAVES_FICHA) <= set(ficha) and all(len(str(ficha[k]).strip()) >= 15 for k in CLAVES_FICHA)),
        ("ficha: el desempeño trae cifras y el uso previsto dice qué NO hacer",
         lambda: any(ch.isdigit() for ch in ficha["desempeno"])
                 and re.search(r"\bno\b", sin_tildes(ficha["uso_previsto_y_no_recomendados"])) is not None),
        ("ficha: las limitaciones son concretas (≥ 40 caracteres, no 'no hay')",
         lambda: len(ficha["limitaciones"].strip()) >= 40
                 and sin_tildes(ficha["limitaciones"]).strip(" .") not in {"no hay", "ninguna", "no hay limitaciones"}),
        ("conclusion_ej4 es la correcta",
         lambda: conclusion_ej4 == "b"),
    ]
    return ejecutar_checks(checks)

verificar_ej4()

### 4.3 Extra (opcional) · ✏️ Gráfico de tasas por subgrupo
Gráfico de barras con `fpr` y `fnr` por sexo y sus intervalos como barras de error. Guárdenlo con `guardar_figura(fig4, "lab13_tasas_por_sexo.png")`.

In [ ]:
# ✏️ TU TURNO (opcional)
# fig4, ax4 = ...
# guardar_figura(fig4, "lab13_tasas_por_sexo.png")

In [ ]:
def verificar_ej4_extra():
    if not (FIGS / "lab13_tasas_por_sexo.png").exists():
        print("⊘ extra no realizado (no cuenta como falla)")
        return True
    return ejecutar_checks([("existe figs/lab13_tasas_por_sexo.png",
                             lambda: (FIGS / "lab13_tasas_por_sexo.png").stat().st_size > 0)])

verificar_ej4_extra()

---
## Ahora con SU proyecto — la prueba del clon vacío (11:55–12:10)

Lo que sigue no es un ejercicio: son dos tablas para copiar al informe o al README de su repositorio. Mientras trabajan, pasa la **última ronda de consulta** de los Informes 3+4.

**1. Prueba del clon vacío (reproducibilidad).** Clonen su repositorio en una **carpeta nueva**, sigan **su propio README** y corran los **cuatro notebooks en orden**. Para cada punto: `cumple` / `no cumple` y `qué hago`. Es el gate de los Informes 3+4.

| Punto | Cumple / no cumple | Qué hago |
|---|---|---|
| El clon vacío funciona siguiendo el README | | |
| `requirements.txt` con versiones | | |
| Semilla fija | | |
| El proceso se ejecuta con un comando | | |
| Datos documentados (no subidos si pesan o son sensibles) | | |
| Al menos tres pruebas que pasan | | |
| README con resultados con su origen y limitaciones | | |
| Declaración de uso de IA | | |
| Commits con mensajes claros | | |

**2. Borrador de ética, privacidad y sesgo (para la casa).** Una a tres frases por pregunta, con la evidencia (celda o cifra). *"No aplica, porque…"* es una respuesta válida.

| Pregunta | Respuesta (1 a 3 frases) | Evidencia (celda o cifra) |
|---|---|---|
| ¿Qué datos personales o sensibles contiene el dataset y cómo se trataron? | | |
| ¿Para qué fin se publicaron y para qué los usamos? | | |
| ¿Qué riesgo de reidentificación hay y qué se hizo? | | |
| ¿El modelo se comporta distinto entre subgrupos, con qué intervalo? | | |
| ¿Quién podría verse perjudicado por un error del modelo y qué control humano hay? | | |

> **Informes 3+4 (documento único): domingo 08-nov, 23:59, AULA-USM.** −10 por día de atraso, máx. 3, aplicado a las dos notas. Esta es la última ronda de consulta.

---
### Checklist de salida de la Clase 13

- [ ] Ejercicios 1, 2 y 3 pasan su verificación automática (el 4, la ficha y los extras pueden quedar para la casa)
- [ ] Tabla de reproducibilidad completa sobre el repositorio del proyecto: cada punto con `cumple` y `qué hago`
- [ ] Borrador de las cinco respuestas de ética, privacidad y sesgo (para la casa)
- [ ] Lo que no se alcanza a corregir hoy, anotado como limitación para el informe
- [ ] Informes 3+4 (documento único): **domingo 8-nov, 23:59, AULA-USM**
- [ ] Para la clase de reportes y comunicación: traer el resultado principal del proyecto y la figura que lo muestra
- [ ] Recordatorio: **Informe 5 y exposición, 03-dic**